In [2]:
# import library
import numpy as np # perhitungan
import skfuzzy as fuzz # scikit-fuzzy
from skfuzzy import control as ctrl # control system

In [3]:
# Input
distance = ctrl.Antecedent(np.arange(0, 101, 1), 'distance')  # meters (0m-100m)
rel_speed = ctrl.Antecedent(np.arange(-50, 51, 1), 'relative_speed')  # km/h (-50km/h(mendekat) sampai 50km/h(menjauh))
slope = ctrl.Antecedent(np.arange(-10, 11, 1), 'slope')  # degrees (-10 deg(turun) sampai 10 deg(naik))
road_condition = ctrl.Antecedent(np.arange(0, 11, 1), 'road_condition')  # (0=slippery, 10=rough)

# Output
acceleration = ctrl.Consequent(np.arange(-5, 6, 1), 'acceleration')  # -5 (hard brake) to +5 (hard accel)

In [31]:
# Membership Functions

# Distance
distance['dekat'] = fuzz.trapmf(distance.universe, [0, 0, 15, 30]) #(0-30m)
distance['sedang'] = fuzz.trimf(distance.universe, [20, 50, 80]) #(20-80m)
distance['jauh'] = fuzz.trapmf(distance.universe, [60, 80, 100, 100]) #(60-100m)

# Relative Speed
rel_speed['mendekat_cepat'] = fuzz.trapmf(rel_speed.universe, [-50, -50, -20, -5]) #(-50km/h sampai -5km/h)
rel_speed['stabil'] = fuzz.trimf(rel_speed.universe, [-10, 0, 10]) #(-10km/h sampai 10km/h)
rel_speed['menjauh'] = fuzz.trapmf(rel_speed.universe, [5, 20, 50, 50]) #(5km/h sampai 50km/h)

# Slope
slope['turun'] = fuzz.trapmf(slope.universe, [-10, -10, -5, -1]) #(-10deg sampai -1deg)
slope['datar'] = fuzz.trimf(slope.universe, [-2, 0, 3]) #(-2deg sampai 3deg)
slope['naik'] = fuzz.trapmf(slope.universe, [2, 5, 10, 10]) #(2deg sampai 10deg)

# Road Condition
road_condition['licin'] = fuzz.trapmf(road_condition.universe, [0, 0, 2, 4])
road_condition['normal'] = fuzz.trimf(road_condition.universe, [3, 5, 7])
road_condition['kasar'] = fuzz.trapmf(road_condition.universe, [6, 8, 10, 10])

# Acceleration (Output)
acceleration['rem_keras'] = fuzz.trapmf(acceleration.universe, [-5, -5, -3.5, -2.5])
acceleration['rem'] = fuzz.trimf(acceleration.universe, [-3.0, -1.5, 0])
acceleration['pertahankan'] = fuzz.trimf(acceleration.universe, [-1.0, 0, 1.0])
acceleration['akselerasi'] = fuzz.trimf(acceleration.universe, [0, 1.5, 3.0])
acceleration['akselerasi_keras'] = fuzz.trapmf(acceleration.universe, [2.5, 3.5, 5, 5])

In [34]:
# Rule Base Design 

# ATURAN DASAR JARAK & KECEPATAN RELATIF 
rule1 = ctrl.Rule(distance['dekat'] & rel_speed['mendekat_cepat'], acceleration['rem_keras'])
rule2 = ctrl.Rule(distance['dekat'] & rel_speed['stabil'], acceleration['rem'])
rule3 = ctrl.Rule(distance['dekat'] & rel_speed['menjauh'], acceleration['pertahankan']) 
rule4 = ctrl.Rule(distance['sedang'] & rel_speed['mendekat_cepat'], acceleration['rem'])
rule5 = ctrl.Rule(distance['sedang'] & rel_speed['stabil'], acceleration['pertahankan'])
rule6 = ctrl.Rule(distance['sedang'] & rel_speed['menjauh'], acceleration['akselerasi'])
rule7 = ctrl.Rule(distance['jauh'] & rel_speed['mendekat_cepat'], acceleration['pertahankan'])  
rule8 = ctrl.Rule(distance['jauh'] & rel_speed['stabil'], acceleration['pertahankan'])
rule9 = ctrl.Rule(distance['jauh'] & rel_speed['menjauh'], acceleration['akselerasi_keras'])

# ATURAN KOMBINASI KEMIRINGAN JALAN
rule10 = ctrl.Rule(slope['naik'] & rel_speed['stabil'], acceleration['akselerasi'])
rule11 = ctrl.Rule(slope['naik'] & rel_speed['menjauh'], acceleration['akselerasi_keras'])  
rule12 = ctrl.Rule(slope['naik'] & rel_speed['mendekat_cepat'], acceleration['rem'])  
rule13 = ctrl.Rule(slope['turun'] & rel_speed['stabil'], acceleration['rem'])
rule14 = ctrl.Rule(slope['turun'] & rel_speed['menjauh'], acceleration['pertahankan'])  
rule15 = ctrl.Rule(slope['turun'] & rel_speed['mendekat_cepat'], acceleration['rem_keras'])  
rule16 = ctrl.Rule(slope['datar'] & rel_speed['stabil'], acceleration['pertahankan']) 
rule17 = ctrl.Rule(slope['datar'] & rel_speed['mendekat_cepat'], acceleration['rem']) 
rule18 = ctrl.Rule(slope['datar'] & rel_speed['menjauh'], acceleration['akselerasi']) 

# ATURAN KONDISI JALAN
rule19 = ctrl.Rule(road_condition['licin'] & rel_speed['mendekat_cepat'], acceleration['rem_keras'])
rule20 = ctrl.Rule(road_condition['licin'] & distance['dekat'], acceleration['rem'])
rule21 = ctrl.Rule(road_condition['licin'] & rel_speed['stabil'], acceleration['rem'])  
rule22 = ctrl.Rule(road_condition['licin'] & slope['turun'], acceleration['rem_keras']) 
rule23 = ctrl.Rule(road_condition['kasar'] & distance['jauh'] & rel_speed['menjauh'], acceleration['pertahankan'])
rule24 = ctrl.Rule(road_condition['kasar'] & rel_speed['mendekat_cepat'], acceleration['rem'])  
rule25 = ctrl.Rule(road_condition['normal'] & slope['naik'] & rel_speed['menjauh'], acceleration['akselerasi_keras'])
rule26 = ctrl.Rule(road_condition['normal'] & distance['sedang'] & rel_speed['stabil'], acceleration['pertahankan'])  

In [35]:
# Fuzzy Control System
acc_ctrl = ctrl.ControlSystem([
    rule1, rule2, rule3, rule4, rule5, rule6, rule7,
    rule8, rule9, rule10, rule11, rule12, rule13, rule14, rule15, rule16, rule17, rule18, rule19,
    rule20, rule21, rule22, rule23, rule24, rule25, rule26
])
acc_sim = ctrl.ControlSystemSimulation(acc_ctrl)

### Output

In [36]:
# Interpretasi Output
def interpret_acceleration(output_value):
    """Mengubah output numerik fuzzy menjadi keputusan manusiawi."""
    if output_value <= -3.2:
        return "REM KERAS (Deselerasi Darurat)"
    elif -3.1 < output_value <= -1.5:
        return "REM (Deselerasi Normal)" 
    elif -1.4 < output_value < 1.5:
        return "PERTAHANKAN (Jaga Kecepatan)"
    elif 1.5 <= output_value < 3.2:
        return "AKSELERASI (Percepat Sedikit)"
    else:
        return "AKSELERASI KERAS (Percepat Penuh)"

In [48]:
# Contoh 1
# Input uji :
acc_sim.input['distance'] = 40        # meter 
acc_sim.input['relative_speed'] = -10 # km/h 
acc_sim.input['slope'] = 2            # derajat 
acc_sim.input['road_condition'] = 5   # normal

# Jalankan perhitungan fuzzy
acc_sim.compute()

# Ambil hasil dan tampilkan
output_value = acc_sim.output['acceleration']
decision = interpret_acceleration(output_value)

print(f"\nHasil perhitungan fuzzy: {output_value:.2f}")
print("Keputusan sistem:", decision)


Hasil perhitungan fuzzy: -1.50
Keputusan sistem: REM (Deselerasi Normal)


In [50]:
# Contoh 2
# Input uji :
acc_sim.input['distance'] = 60        # meter 
acc_sim.input['relative_speed'] = 30 # km/h 
acc_sim.input['slope'] = -2            # derajat 
acc_sim.input['road_condition'] = 7   # kasar

# Jalankan perhitungan fuzzy
acc_sim.compute()

# Ambil hasil dan tampilkan
output_value = acc_sim.output['acceleration']
decision = interpret_acceleration(output_value)

print(f"\nHasil perhitungan fuzzy: {output_value:.2f}")
print("Keputusan sistem:", decision)


Hasil perhitungan fuzzy: 1.17
Keputusan sistem: PERTAHANKAN (Jaga Kecepatan)


In [39]:
# Contoh 3
# Input uji :
acc_sim.input['distance'] = 30        # meter 
acc_sim.input['relative_speed'] = 20 # km/h 
acc_sim.input['slope'] = 4            # derajat 
acc_sim.input['road_condition'] = 5   # kasar

# Jalankan perhitungan fuzzy
acc_sim.compute()

# Ambil hasil dan tampilkan
output_value = acc_sim.output['acceleration']
decision = interpret_acceleration(output_value)

print(f"\nHasil perhitungan fuzzy: {output_value:.2f}")
print("Keputusan sistem:", decision)


Hasil perhitungan fuzzy: 3.03
Keputusan sistem: AKSELERASI (Percepat Sedikit)


In [51]:
# Contoh 4
# Input uji :
acc_sim.input['distance'] = 15        # meter 
acc_sim.input['relative_speed'] = -25 # km/h 
acc_sim.input['slope'] = 2            # derajat 
acc_sim.input['road_condition'] = 5   # normal

# Jalankan perhitungan fuzzy
acc_sim.compute()

# Ambil hasil dan tampilkan
output_value = acc_sim.output['acceleration']
decision = interpret_acceleration(output_value)

print(f"\nHasil perhitungan fuzzy: {output_value:.2f}")
print("Keputusan sistem:", decision)


Hasil perhitungan fuzzy: -3.24
Keputusan sistem: REM KERAS (Deselerasi Darurat)


# Evaluasi dan Diskusi
### Apakah keputusan sistem sesuai dengan logika berkendara di dunia nyata?
Ya, keputusan sistem sudah sesuai dengan logika berekendara di dunia nyata. Fuzzy logic controller menghasilkan output yang mencerminkan perilaku pengemudi dalam berbagai kondisi jalan. Misalnya, ketika kendaraan berada cukup dekat dan mendekati kendaraan lain dengan kecepatan relatif –10 km/jam, sistem memutuskan untuk melakukan pengereman normal. Keputusan tersebut sesuai dengan tindakan pengemudi manusia untuk menjaga jarak aman. Sebaliknya, saat kendaraan menjauh di jalan yang sedikit menurun dan permukaannya kasar, sistem memilih untuk mempertahankan kecepatan agar tetap stabil. Pada kondisi jalan menanjak, sistem memberikan akselerasi bertahap atau kuat tergantung pada jarak dan kecepatan relatif, sebagaimana pengemudi akan menekan pedal gas lebih dalam untuk menyesuaikan dengan tanjakan. Secara keseluruhan, keputusan sistem fuzzy ini sudah mendekati intuisi manusia dalam situasi mengemudi yang sebenarnya.

Meskipun fuzzy logic mampu meniru cara berpikir manusia dan menghasilkan keputusan yang fleksibel, sistem ini tetap sangat bergantung pada rangkaian aturan yang dirancang sebelumnya. Artinya, kualitas keputusan yang dihasilkan sangat ditentukan oleh seberapa baik aturan-aturan tersebut merepresentasikan logika mengemudi yang sebenarnya. Jika aturan yang dibuat tidak konsisten, tumpang tindih, atau tidak mencerminkan kondisi nyata di lapangan, maka hasil keputusan fuzzy logic juga bisa menjadi tidak rasional dan tidak sesuai dengan perilaku manusia. Dengan kata lain, fuzzy logic hanya akan sebaik pengetahuan dan pengalaman yang dimasukkan ke dalam basis aturannya, sehingga perancangan aturan yang tepat dan realistis menjadi hal yang sangat penting agar sistem tetap relevan dan akurat dalam berbagai situasi berkendara.

### Jelaskan bagaimana logika fuzzy menawarkan perilaku kontrol yang lebih fleksibel dan mirip manusia dibandingkan dengan traditional threshold-based systems.
Fuzzy logic menawarkan perilaku kontrol yang lebih fleksibel dan menyerupai cara berpikir manusia dibandingkan traditional threshold-based systems. Dalam threshold-based systems, keputusan biasanya bersifat kaku dan biner. Misalnya, jika jarak < 30 meter maka rem, sedangkan jika > 30 meter maka akselerasi. Namun pada fuzzy logic, setiap variabel seperti jarak, kecepatan relatif, kemiringan jalan, dan kondisi permukaan jalan memiliki tingkat keanggotaan yang tidak mutlak, sehingga sistem dapat mengenali kondisi “agak dekat”, “sedikit menjauh”, atau “menanjak ringan”. Hal ini memungkinkan keputusan yang lebih halus dan adaptif terhadap perubahan kondisi. Selain itu, fuzzy logic mampu menggabungkan beberapa faktor secara bersamaan untuk menghasilkan keputusan yang lebih realistis, mirip seperti cara manusia menimbang berbagai situasi sebelum mempercepat atau mengerem. Dengan demikian, fuzzy logic menciptakan transisi keputusan yang tidak mendadak dan lebih alami, sehingga perilakunya terasa lebih manusiawi dalam mengendalikan kendaraan di berbagai situasi.

In [2]:
pip freeze > requirements_fuzzy.txt

Note: you may need to restart the kernel to use updated packages.
